In [ ]:
# SKKU PubMed Research Lineage — ONE CELL
# 1) SKKU affiliation PubMed 전체 수집 (10k 제한 회피용 연/월 분할 + pagination)
# 2) 전체 seed corpus에서 연구자 profile / coauthor network / research community 생성
# 3) ORCID 기반으로 일부 핵심 연구자의 SKKU 이후 후속 논문 심층 추적
# 4) 결과 CSV/JSON/Markdown/HTML 생성 + 주요 표 표시

import os, sys, subprocess, json
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

# ===== 사용자 설정 =====
START_YEAR = 2010
END_YEAR = 2026
FULL_CRAWL = True        # True: 연/월 분할 pagination으로 기간 전체 수집
MAX_RESULTS = 0          # FULL_CRAWL=True일 때 0=전체, 숫자>0이면 안전 cap
PAGE_SIZE = 500
RESUME_CRAWL = True      # 같은 Colab runtime에서 재실행 시 crawl_checkpoint.json 재사용
RUN_DEEP_ORCID_FOLLOWUP = True
MAX_AUTHORS = 50         # 심층 ORCID 추적 대상 수 (전체 지도에는 제한 없음)
MAX_PER_AUTHOR = 150
TOPIC = ""               # 예: "stroke OR cerebrovascular"; 전체면 ""
TOPIC_THRESHOLD = 0.30
NCBI_EMAIL = os.environ.get("NCBI_EMAIL") or input("NCBI email: ").strip()
NCBI_API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # 있으면 자동 사용

if not NCBI_EMAIL:
    raise ValueError("NCBI_EMAIL이 필요합니다.")

# ===== GitHub 최신 코드 동기화 =====
REPO = Path("/content/Paper_AI_Assistant")
URL = "https://github.com/kimtk94/Paper_AI_Assistant.git"

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q", URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", "origin/main"], check=True)

os.chdir(REPO)

# ===== 실행 전 문법 검사 =====
for script in ["src/skku_pubmed_lineage.py", "src/skku_pubmed_annotation.py", "src/skku_pubmed_seed_profiles.py", "src/skku_pubmed_author_followup.py", "src/skku_pubmed_researcher_network.py", "src/skku_pubmed_research_communities.py"]:
    subprocess.run([sys.executable, "-m", "py_compile", script], check=True)
print("✅ Python syntax check passed")

env = os.environ.copy()
env["NCBI_EMAIL"] = NCBI_EMAIL
if NCBI_API_KEY:
    env["NCBI_API_KEY"] = NCBI_API_KEY

def run_stream(cmd, label, env=None, tail_lines=80):
    """Run a long command while streaming logs; show the real failure tail."""
    print(f"▶ {label}")
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        if len(tail) > tail_lines:
            tail.pop(0)
    rc = proc.wait()
    if rc != 0:
        diagnostic = "\n".join(tail[-tail_lines:])
        raise RuntimeError(
            f"{label} failed with exit code {rc}.\n"
            f"----- last {min(len(tail), tail_lines)} log lines -----\n"
            f"{diagnostic}"
        )
    return rc

# ===== STEP 1: SKKU PubMed graph =====
cmd1 = [
    sys.executable, "src/skku_pubmed_lineage.py",
    "--start-year", str(START_YEAR),
    "--end-year", str(END_YEAR),
    "--max-results", str(MAX_RESULTS),
    "--topic-threshold", str(TOPIC_THRESHOLD),
    "--skip-citations",
    "--output-dir", "outputs/skku_pubmed_lineage",
]
if FULL_CRAWL:
    cmd1 += ["--all-results", "--page-size", str(PAGE_SIZE)]
    if RESUME_CRAWL:
        cmd1 += ["--resume"]
if TOPIC.strip():
    cmd1 += ["--topic", TOPIC.strip()]

print("\n=== STEP 1/5: SKKU PubMed full crawl ===")
run_stream(cmd1, "STEP 1/5 SKKU PubMed full crawl", env=env)

# ===== STEP 2: full seed corpus -> researcher profiles =====
print("\n=== STEP 2/5: Full-corpus researcher profiles ===")
subprocess.run([
    sys.executable, "src/skku_pubmed_seed_profiles.py",
    "--papers-json", "outputs/skku_pubmed_lineage/papers.json",
    "--output-dir", "outputs/skku_pubmed_seed_map",
], check=True, env=env)

# ===== STEP 3: scalable observed-edge researcher network =====
print("\n=== STEP 3/5: SKKU-wide researcher network ===")
subprocess.run([
    sys.executable, "src/skku_pubmed_researcher_network.py",
    "--input-dir", "outputs/skku_pubmed_seed_map",
    "--skip-thematic",
    "--topic-threshold", str(TOPIC_THRESHOLD),
], check=True, env=env)

# ===== STEP 4: research communities =====
print("\n=== STEP 4/5: SKKU-wide research communities ===")
subprocess.run([
    sys.executable, "src/skku_pubmed_research_communities.py",
    "--input-dir", "outputs/skku_pubmed_seed_map",
    "--min-edge-score", "0.65",
    "--strong-edge-score", "0.85",
], check=True, env=env)

# ===== STEP 5: optional ORCID deep follow-up =====
if RUN_DEEP_ORCID_FOLLOWUP:
    print("\n=== STEP 5/5: ORCID deep continuation ===")
    cmd2 = [
        sys.executable, "src/skku_pubmed_author_followup.py",
        "--seed-json", "outputs/skku_pubmed_lineage/papers.json",
        "--start-year", "2002",
        "--end-year", str(END_YEAR),
        "--max-authors", str(MAX_AUTHORS),
        "--max-per-author", str(MAX_PER_AUTHOR),
        "--with-citations",
        "--max-citation-checks", "300",
        "--output-dir", "outputs/skku_pubmed_followup",
    ]
    subprocess.run(cmd2, check=True, env=env)
else:
    print("\n=== STEP 5/5: ORCID deep continuation skipped ===")

# ===== 빈 CSV도 안전하게 읽기 =====
def safe_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

papers = safe_csv("outputs/skku_pubmed_lineage/papers.csv")
edges = safe_csv("outputs/skku_pubmed_lineage/edges.csv")
full_researchers = safe_csv("outputs/skku_pubmed_seed_map/researcher_summary.csv")
researcher_network = safe_csv("outputs/skku_pubmed_seed_map/researcher_network_edges.csv")
researcher_nodes = safe_csv("outputs/skku_pubmed_seed_map/researcher_network_nodes.csv")
communities = safe_csv("outputs/skku_pubmed_seed_map/research_communities.csv")
community_members = safe_csv("outputs/skku_pubmed_seed_map/researcher_community_membership.csv")
community_edges = safe_csv("outputs/skku_pubmed_seed_map/community_edges.csv")

registry = safe_csv("outputs/skku_pubmed_followup/author_registry.csv")
followups = safe_csv("outputs/skku_pubmed_followup/lineage_papers.csv")
continuation = safe_csv("outputs/skku_pubmed_followup/continuation_edges.csv")
lineage_links = safe_csv("outputs/skku_pubmed_followup/lineage_edges.csv")
annotations = safe_csv("outputs/skku_pubmed_followup/paper_annotations.csv")
researchers = safe_csv("outputs/skku_pubmed_followup/researcher_summary.csv")

print("\n================ RESULT ================")
print(f"SKKU verified papers : {len(papers):,}")
print(f"SKKU graph edges     : {len(edges):,}")
print(f"Full-map researchers : {len(full_researchers):,}")
print(f"Full-map links       : {len(researcher_network):,}")
print(f"Research communities : {len(communities):,}")
print(f"Deep tracked authors : {len(registry):,}")
print(f"Follow-up papers     : {len(followups):,}")
print(f"Continuation edges   : {len(continuation):,}")
print(f"Research lineage     : {len(lineage_links):,}")
print(f"Annotated papers     : {len(annotations):,}")
print(f"Researcher nodes     : {len(researcher_nodes):,}")
print(f"Deep trajectories    : {len(researchers):,}")

if not edges.empty and "relation" in edges:
    print("\n[SKKU edge types]")
    display(edges["relation"].value_counts().rename_axis("relation").reset_index(name="count"))

if not full_researchers.empty:
    print("\n[Full-corpus SKKU researchers]")
    cols = [c for c in [
        "name", "orcid", "confidence", "paper_count", "first_year", "last_year",
        "top_diseases", "top_methods", "top_data_types", "stage_path"
    ] if c in full_researchers.columns]
    display(full_researchers[cols].head(200))

if not registry.empty:
    print("\n[Deep ORCID tracked researchers]")
    cols = [c for c in ["name", "orcid", "confidence", "seed_pmids"] if c in registry.columns]
    display(registry[cols].head(30))

if not followups.empty:
    print("\n[Researcher publication continuation]")
    cols = [c for c in [
        "year", "tracked_authors", "title", "disease_terms", "methods",
        "data_types", "research_stage", "research_question",
        "journal", "skku_current", "pubmed_url"
    ] if c in followups.columns]
    display(followups[cols].sort_values(["tracked_authors", "year"], ascending=[True, True]).head(100))

if not annotations.empty:
    print("\n[Paper research profiles]")
    cols = [c for c in [
        "pmid", "disease_terms", "methods", "data_types",
        "research_stage", "research_question"
    ] if c in annotations.columns]
    display(annotations[cols].head(100))

if not researchers.empty:
    print("\n[Researcher trajectories]")
    cols = [c for c in [
        "name", "orcid", "paper_count", "first_year", "last_year",
        "top_diseases", "top_methods", "top_data_types",
        "stage_path", "strong_lineage_count", "trajectory_summary"
    ] if c in researchers.columns]
    display(researchers[cols].head(100))

if not lineage_links.empty:
    print("\n[Research lineage: direct citation > ORCID-only]")
    cols = [c for c in [
        "score", "relation", "source", "target", "tracked_authors",
        "source_stage", "target_stage", "progression", "evidence"
    ] if c in lineage_links.columns]
    display(lineage_links[cols].sort_values("score", ascending=False).head(100))

if not researcher_network.empty:
    print("\n[Researcher network: collaboration / citation / thematic overlap]")
    cols = [c for c in [
        "score", "relation", "source_name", "target_name", "shared_papers",
        "direct_citations", "topic_similarity", "shared_diseases",
        "shared_methods", "shared_data_types", "evidence"
    ] if c in researcher_network.columns]
    display(researcher_network[cols].sort_values("score", ascending=False).head(100))

if not communities.empty:
    print("\n[Research communities]")
    cols = [c for c in [
        "community_id", "label", "researcher_count", "hub_researcher",
        "total_papers", "first_year", "last_year", "top_diseases",
        "top_methods", "top_data_types", "stage_path",
        "strong_edge_count", "evidence_summary"
    ] if c in communities.columns]
    display(communities[cols].head(100))

if not community_members.empty:
    print("\n[Community membership / hub researchers]")
    cols = [c for c in [
        "community_id", "researcher_name", "orcid", "paper_count",
        "weighted_degree", "within_community_degree", "role"
    ] if c in community_members.columns]
    display(community_members[cols].head(200))

if not continuation.empty:
    print("\n[Raw continuation/citation edges]")
    display(continuation.head(100))

# ===== 결과 ZIP =====
zip_base = "/content/SKKU_PubMed_Lineage_results"
subprocess.run(
    ["zip", "-qr", zip_base + ".zip",
     "outputs/skku_pubmed_lineage", "outputs/skku_pubmed_seed_map", "outputs/skku_pubmed_followup"],
    check=True
)
print(f"\n📦 ZIP: {zip_base}.zip")
print("🌐 Interactive graph: /content/Paper_AI_Assistant/outputs/skku_pubmed_lineage/lineage.html")
print("📄 Research chains: /content/Paper_AI_Assistant/outputs/skku_pubmed_followup/continuation.md")
print("🧑‍🔬 SKKU-wide researcher network: /content/Paper_AI_Assistant/outputs/skku_pubmed_seed_map/researcher_network.html")
print("🧩 SKKU-wide research community map: /content/Paper_AI_Assistant/outputs/skku_pubmed_seed_map/research_community_map.html")
print("🧭 Deep ORCID trajectory HTML: /content/Paper_AI_Assistant/outputs/skku_pubmed_followup/research_trajectory.html")

# Colab에서 interactive HTML 바로 표시
html_path = Path("outputs/skku_pubmed_lineage/lineage.html")
if html_path.exists():
    display(HTML("<b>완료:</b> 왼쪽 Files에서 <code>lineage.html</code> 또는 결과 ZIP을 열면 됩니다."))
